In [13]:
import os
import cv2
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm

def get_crowd_density_for_all_sets(base_folder="filtered_frames", fps=1): #matches our frame extraction rate
    """
    Calculates the 'Crowd Density' (Edge Intensity) using a Sobel filter 
    for all extracted 2-FPS frames across all DJ sets.
    """
    # Get all the DJ folders (ignoring hidden files like .DS_Store)
    dj_folders = [f for f in os.listdir(base_folder) if not f.startswith('.')]
    
    all_density_data = []

    for dj_name in tqdm(dj_folders, desc="Processing DJ Sets"):
        dj_path = os.path.join(base_folder, dj_name)
        if not os.path.isdir(dj_path): continue
            
        # Get all frames and sort them
        frames = [f for f in os.listdir(dj_path) if f.endswith(('.png', '.jpg'))]
        frames.sort()
        
        if len(frames) == 0:
            continue
            
        print(f"Starting Density Calc: {dj_name} ({len(frames)} frames)")
        
        for frame_file in frames:
            # Safely grab the frame number "frame_0150.jpg" -> 150
            # If your names are different, we might need a regex, but this usually works:
            try:
                # Extracts the digits right before the .jpg
                current_frame_num = int(frame_file.split('_')[-1].split('.')[0])
            except ValueError:
                continue
                
            frame_path = os.path.join(dj_path, frame_file)
            
            # 1. Load frame in grayscale
            gray = cv2.imread(frame_path, cv2.IMREAD_GRAYSCALE)
            if gray is None: continue
                
            # Resize it down to speed up the math (we don't need 1080p for density edge detection)
            gray = cv2.resize(gray, (640, 360))
            
            # 2. Blur slightly to remove extreme camera static noise, but keep hard edges (arms/heads)
            blurred = cv2.GaussianBlur(gray, (5, 5), 0)
            
            # 3. Apply Sobel filter to find horizontal and vertical edges
            grad_x = cv2.Sobel(blurred, cv2.CV_64F, 1, 0, ksize=3)
            grad_y = cv2.Sobel(blurred, cv2.CV_64F, 0, 1, ksize=3)
            
            # 4. Calculate absolute magnitude of the edges
            abs_grad_x = cv2.convertScaleAbs(grad_x)
            abs_grad_y = cv2.convertScaleAbs(grad_y)
            edge_img = cv2.addWeighted(abs_grad_x, 0.5, abs_grad_y, 0.5, 0)
            
            # 5. The 'density' score is the average edge intensity per pixel!
            total_pixels = edge_img.shape[0] * edge_img.shape[1]
            density_score = np.sum(edge_img) / total_pixels
            
            all_density_data.append({
                'video_name': dj_name,
                'timestamp': current_frame_num / fps, # Convert frame num back to seconds
                'density_score': density_score
            })

    # Return the final organized DataFrame
    df = pd.DataFrame(all_density_data)
    
    # Optional: smooth out the score over a 10-second rolling window so it doesn't jump too much
    # df['density_score'] = df.groupby('video_name')['density_score'].transform(lambda x: x.rolling(5, min_periods=1).mean())
    
    return df

# --- RUN THE CODE ---
# density_df = get_crowd_density_for_all_sets(base_folder="filtered_frames", fps=2)
# print(density_df.head(10))
# density_df.to_csv("crowd_density_metrics.csv", index=False)


In [14]:
density_df = get_crowd_density_for_all_sets(base_folder="filtered_frames", fps=2)
print(density_df.head(10))
density_df.to_csv("crowd_density_metrics.csv", index=False)


Processing DJ Sets:   0%|          | 0/13 [00:00<?, ?it/s]

Starting Density Calc: Yaeji  Boiler Room New York - Boiler Room (720p, h264) (806 frames)
Starting Density Calc: Funky Disco & House Mix Inside a Circus  Felix da Housecat - Book Club Radio (720p, h264, youtube) (454 frames)
Starting Density Calc: Yung Singh  Boiler Room Melbourne - Boiler Room (720p, h264, youtube) (497 frames)
Starting Density Calc: ALISHA Groovy Tech-House DJ Set Live From DJ Mag HQ - DJ Mag (720p, h264, youtube) (894 frames)
Starting Density Calc: Fred again.. ｜ Boiler Room： London (9160 frames)
Starting Density Calc: KETTAMA  Boiler Room London - Boiler Room (720p, h264) (267 frames)
Starting Density Calc: Marlon Hoffstadt  Boiler Room Melbourne - Boiler Room (720p, h264) (10032 frames)
Starting Density Calc: AZYR  Boiler Room x Teletech Festival 2023 - Boiler Room (720p, h264) (592 frames)
Starting Density Calc: ¥ØUUK€ ¥UK1MATU  Boiler Room Tokyo - Boiler Room (720p, h264) (470 frames)
Starting Density Calc: Chase & Status  Boiler Room London - Boiler Room (720p

In [16]:
# we borrowed the has_dj_equipment 
def has_dj_equipment(frame, threshold=0.04):
    """
    Checks if the lower half of the frame contains a high density of hard, 
    geometric edges (a dead giveaway for DJ mixers, CDJs, and laptops).
    """
    # 1. Grab the dimensions of the image
    height, width = frame.shape[:2]
    
    # 2. Slice the image perfectly in half and KEEP ONLY THE BOTTOM HALF!
    lower_half = frame[height//2:, :]
    
    # 3. Convert that bottom half to grayscale for edge detection
    gray_half = cv2.cvtColor(lower_half, cv2.COLOR_BGR2GRAY)
    
    # 4. Use the Canny Edge Detector to find all the hard, contrasting lines
    # (Mixer knobs, straight edges of laptops, glowing buttons)
    edges = cv2.Canny(gray_half, threshold1=50, threshold2=150)
    
    # 5. Calculate the "Edge Density" (what percentage of the bottom half is hard lines?)
    total_pixels = edges.shape[0] * edges.shape[1]
    edge_pixels = np.count_nonzero(edges)
    
    density = edge_pixels / total_pixels
    
    # If the density of hard lines is greater than the 4% threshold, it's equipment!
    return density > threshold

def filter_frames(frames_dir, output_dir, threshold=0.04):
    os.makedirs(output_dir, exist_ok=True)
    kept, total = 0, 0

    for fname in sorted(os.listdir(frames_dir)):
        if not fname.endswith('.jpg'):
            continue
        path = os.path.join(frames_dir, fname)
        frame = cv2.imread(path)
        if frame is None:          # guard against unreadable files too
            continue
        total += 1
        
        if has_dj_equipment(frame, threshold):
            cv2.imwrite(os.path.join(output_dir, fname), frame)
            kept += 1

    if total == 0:
        print(f"No frames found in {frames_dir}")
        return
    print(f"Kept {kept}/{total} frames ({100*kept/total:.1f}%)")


In [20]:
import os
import glob
import pandas as pd

def process_all_dj_sets(video_dir="dj_sets_videos", target_fps=5):
    """
    Loops through every .mp4 file in the video directory, calculates 
    the motion energy, and combines everything into one master DataFrame.
    """
    # Find all .mp4 files in the directory
    video_files = glob.glob(os.path.join(video_dir, "*.mp4"))
    
    all_motion_data = []
    
    print(f"Found {len(video_files)} DJ sets to process.")
    
    for video_path in video_files:
        # Extract just the video name without the folder path or .mp4 extension
        video_name = os.path.splitext(os.path.basename(video_path))[0]
        
        # Run our awesome on-the-fly Lucas-Kanade function
        df_single_video = analyze_video_motion_lk(video_path, target_fps=target_fps)
        
        if df_single_video is not None and not df_single_video.empty:
            # Add a column so we know which video this data belongs to
            df_single_video.insert(0, 'video_name', video_name)
            
            all_motion_data.append(df_single_video)
            print(f"✅ Finished: {video_name}")
        else:
            print(f"❌ Failed to process: {video_name}")
            
    # Combine all the individual dataframes into one massive dataframe
    if all_motion_data:
        master_df = pd.concat(all_motion_data, ignore_index=True)
        return master_df
    else:
        print("No data was processed.")
        return pd.DataFrame()

# --- RUN THE CODE ---
# Warning: This might take a little while depending on how many videos you have!
video_directory = "/Users/fokia/CentraleSupelec-CV-and-RL-project/dj_sets_videos"
master_motion_df = process_all_dj_sets(video_directory, target_fps=5)

# Save the results to a CSV so you never have to run this heavy compute again!
if not master_motion_df.empty:
    master_motion_df.to_csv("all_motion_metrics_lk.csv", index=False)
    print("\n🎉 Success! All motion data saved to all_motion_metrics_lk.csv")
    print(master_motion_df.head(10))


Found 13 DJ sets to process.
Analyzing Motion on: /Users/fokia/CentraleSupelec-CV-and-RL-project/dj_sets_videos/Funky Disco & House Mix Inside a Circus  Felix da Housecat - Book Club Radio (720p, h264, youtube).mp4


  0%|          | 0/164657 [00:00<?, ?it/s]

✅ Finished: Funky Disco & House Mix Inside a Circus  Felix da Housecat - Book Club Radio (720p, h264, youtube)
Analyzing Motion on: /Users/fokia/CentraleSupelec-CV-and-RL-project/dj_sets_videos/ALISHA Groovy Tech-House DJ Set Live From DJ Mag HQ - DJ Mag (720p, h264, youtube).mp4


  0%|          | 0/93411 [00:00<?, ?it/s]

✅ Finished: ALISHA Groovy Tech-House DJ Set Live From DJ Mag HQ - DJ Mag (720p, h264, youtube)
Analyzing Motion on: /Users/fokia/CentraleSupelec-CV-and-RL-project/dj_sets_videos/KETTAMA  Boiler Room London - Boiler Room (720p, h264).mp4


...

✅ Finished: ¥ØUUK€ ¥UK1MATU  Boiler Room Tokyo - Boiler Room (720p, h264)
Analyzing Motion on: /Users/fokia/CentraleSupelec-CV-and-RL-project/dj_sets_videos/Yaeji  Boiler Room New York - Boiler Room (720p, h264).mp4


[h264 @ 0x1069271c0] Invalid NAL unit size (8944940 > 24839).
[h264 @ 0x1069271c0] missing picture in access unit with size 24843
[h264 @ 0x106a2abf0] Invalid NAL unit size (8944940 > 24839).
[h264 @ 0x106a2abf0] Error splitting the input into NAL units.
[h264 @ 0x1069271c0] Invalid NAL unit size (1788368697 > 6713).
[h264 @ 0x1069271c0] missing picture in access unit with size 6717
[h264 @ 0x106a33770] Invalid NAL unit size (1788368697 > 6713).
[h264 @ 0x106a33770] Error splitting the input into NAL units.
[h264 @ 0x1069271c0] Invalid NAL unit size (-1899009465 > 1856).
[h264 @ 0x1069271c0] missing picture in access unit with size 1860
[h264 @ 0x106a3c2f0] Invalid NAL unit size (-1899009465 > 1856).
[h264 @ 0x106a3c2f0] Error splitting the input into NAL units.
[h264 @ 0x1069271c0] Invalid NAL unit size (1171480431 > 9091).
[h264 @ 0x1069271c0] missing picture in access unit with size 9095
[h264 @ 0x106a44e70] Invalid NAL unit size (1171480431 > 9091).
[h264 @ 0x106a44e70] Error split

  0%|          | 0/118511 [00:00<?, ?it/s]

[h264 @ 0x10692fc50] Invalid NAL unit size (31480702 > 24883).
[h264 @ 0x10692fc50] missing picture in access unit with size 24887
[h264 @ 0x1069680d0] Invalid NAL unit size (31480702 > 24883).
[h264 @ 0x1069680d0] Error splitting the input into NAL units.
[h264 @ 0x10692fc50] Invalid NAL unit size (777023978 > 9905).
[h264 @ 0x10692fc50] missing picture in access unit with size 9909
[h264 @ 0x106970c50] Invalid NAL unit size (777023978 > 9905).
[h264 @ 0x106970c50] Error splitting the input into NAL units.
[h264 @ 0x10692fc50] Invalid NAL unit size (843580276 > 2717).
[h264 @ 0x10692fc50] missing picture in access unit with size 2721
[h264 @ 0x1069797d0] Invalid NAL unit size (843580276 > 2717).
[h264 @ 0x1069797d0] Error splitting the input into NAL units.
[h264 @ 0x10692fc50] Invalid NAL unit size (-1950439613 > 6855).
[h264 @ 0x10692fc50] missing picture in access unit with size 6859
[h264 @ 0x106982350] Invalid NAL unit size (-1950439613 > 6855).
[h264 @ 0x106982350] Error splitti

✅ Finished: Yaeji  Boiler Room New York - Boiler Room (720p, h264)

🎉 Success! All motion data saved to all_motion_metrics_lk.csv
                                          video_name      time     motion
0  Funky Disco & House Mix Inside a Circus  Felix...  0.000000   0.000000
1  Funky Disco & House Mix Inside a Circus  Felix...  0.166833   1.134673
2  Funky Disco & House Mix Inside a Circus  Felix...  0.333667  22.147217
3  Funky Disco & House Mix Inside a Circus  Felix...  0.500500   0.860952
4  Funky Disco & House Mix Inside a Circus  Felix...  0.667333   0.284468
5  Funky Disco & House Mix Inside a Circus  Felix...  0.834167   0.181580
6  Funky Disco & House Mix Inside a Circus  Felix...  1.001000   0.212334
7  Funky Disco & House Mix Inside a Circus  Felix...  1.167833   0.200289
8  Funky Disco & House Mix Inside a Circus  Felix...  1.334667   0.218475
9  Funky Disco & House Mix Inside a Circus  Felix...  1.501500   0.151652


In [21]:
master_motion_df

,video_name,time,motion
0,Funky Disco & House Mix Inside a Circus Felix...,0.000000,0.000000
1,Funky Disco & House Mix Inside a Circus Felix...,0.166833,1.134673
2,Funky Disco & House Mix Inside a Circus Felix...,0.333667,22.147217
3,Funky Disco & House Mix Inside a Circus Felix...,0.500500,0.860952
4,Funky Disco & House Mix Inside a Circus Felix...,0.667333,0.284468
...,...,...,...
241138,Yaeji Boiler Room New York - Boiler Room (720...,2050.715333,12.779406
241139,Yaeji Boiler Room New York - Boiler Room (720...,2050.882167,9.506636
241140,Yaeji Boiler Room New York - Boiler Room (720...,2051.049000,8.276356
241141,Yaeji Boiler Room New York - Boiler Room (720...,2051.215833,13.066569


In [3]:
#test it
df = get_motion_lk("/Users/fokia/CentraleSupelec-CV-and-RL-project/dj_sets_videos/AZYR  Boiler Room x Teletech Festival 2023 - Boiler Room (720p, h264).mp4")

Processed 6000 / 89679 frames...
Processed 12000 / 89679 frames...
Processed 18000 / 89679 frames...
Processed 24000 / 89679 frames...
Processed 30000 / 89679 frames...
Processed 36000 / 89679 frames...
Processed 42000 / 89679 frames...
Processed 48000 / 89679 frames...
Processed 54000 / 89679 frames...
Processed 60000 / 89679 frames...
Processed 66000 / 89679 frames...
Processed 72000 / 89679 frames...
Processed 78000 / 89679 frames...
Processed 84000 / 89679 frames...


In [5]:
df

,time,motion
0,0.48,0.000000
1,0.96,4.707239
2,1.44,0.804446
3,1.92,1.200536
4,2.40,1.002996
...,...,...
7468,3585.12,45.877670
7469,3585.60,55.266350
7470,3586.08,43.051849
7471,3586.56,52.210735
